In [1]:
# %% [code] Vorhersage auf kombiniertem Datensatz (Train, Test, Inference)
from pathlib import Path
import pandas as pd
from joblib import load
from FIT_python.config import RESULTS_DATA_DIR

# 1) Pfade konfigurieren
species = "eurasian_otter"
splits_dir = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/data/splits") / species
output_csv = Path("C:/Users/Frede/OneDrive/Projects and Disschaptors/FIT_package/results/data") / f"{species}_all_predictions.csv"
maj_path   = Path(RESULTS_DATA_DIR) / "random_search_standard_metrics" / "best_majority"  / f"{species}.joblib"
bal_path   = Path(RESULTS_DATA_DIR) / "random_search_standard_metrics" / "best_balanced"   / f"{species}.joblib"

# 2) Modelle laden
best_maj = load(maj_path)
best_bal = load(bal_path)

# 3) Parquet-Dateien einlesen und data_origin anhängen
dfs = []
for split in ("train", "test", "inference"):
    df_split = pd.read_parquet(splits_dir / f"{split}.parquet")
    df_split["data_origin"] = split
    dfs.append(df_split)
df_all = pd.concat(dfs, ignore_index=True)

# 4) Numerische Features extrahieren
#    Wir verwenden die feature_cols des ersten Transformers der Pipeline
feature_cols = best_maj.named_steps["transform"].feature_cols
X_all = df_all[feature_cols].copy()

# 5) Vorhersagen auf dem kombinierten Datensatz
pred_maj_num = best_maj.predict(X_all)
pred_bal_num = best_bal.predict(X_all)

# 6) Numerische Labels zu Buchstaben mappen
label_map = {0: "f", 1: "m"}
df_all["pred_majority_num"]  = pred_maj_num
df_all["pred_balanced_num"]  = pred_bal_num
df_all["pred_maj_label"]     = df_all["pred_majority_num"].map(label_map)
df_all["pred_bal_label"]     = df_all["pred_balanced_num"].map(label_map)

# 7) Ergebnis abspeichern
df_all.to_csv(output_csv, index=False)
print(f"Alle Vorhersagen (Train/Test/Inference) gespeichert in {output_csv}")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Frede\\OneDrive\\Projects and Disschaptors\\FIT_package\\results\\data\\random_search_standard_metrics\\best_majority\\eurasian_otter.joblib'